## Structured Output

Models can be requested to provide their response in a format matching a given Schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing strcutured output

#### Pydantic

In [38]:
import os 
from dotenv import load_dotenv
load_dotenv()

True

In [39]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="llama-3.3-70b-versatile")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024D136531D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024D13594790>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [40]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        json_schema_extra={"description": "The title of the movie"}
    )
    year: int = Field(
        json_schema_extra={"description": "The year the movie was released"}
    )
    director: str = Field(
        json_schema_extra={"description": "The director of the movie"}
    )
    rating: float = Field(
        json_schema_extra={"description": "The movie rating out of 10"}
    )

In [41]:

llm_with_structure=llm.with_structured_output(Movie)
llm_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 32768, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x0000024D136531D0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x0000024D13594790>, model_name='llama-3.3-70b-versatile', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The movie rating out of 10', 'type': 'number'}}, 'required': ['title', '

In [42]:
llm.invoke("Provide the detals about the movie inception")

AIMessage(content='**Inception (2010) Movie Details**\n\n**Overview**\n\nInception is a science fiction action film written, produced, and directed by Christopher Nolan. The movie explores the concept of shared dreaming, where a team of thieves navigate the subconscious minds of their targets to steal secrets or plant ideas.\n\n**Plot**\n\nThe film follows Cobb (Leonardo DiCaprio), a skilled thief who specializes in entering people\'s dreams and stealing their secrets. Cobb is hired by a wealthy businessman named Saito (Ken Watanabe) to perform a task known as "inception" - planting an idea in someone\'s mind instead of stealing one.\n\nSaito wants Cobb to convince Robert Fischer (Cillian Murphy), the son of a dying business magnate, to dissolve his father\'s company. In return, Saito promises to clear Cobb\'s name, which is wanted by the authorities, and allow him to return to the United States to see his children.\n\nCobb assembles a team of experts to help him with the task:\n\n1. A

In [43]:
response = llm_with_structure.invoke("Provide the details about the movie Inception")

In [44]:
print(response)

title='Inception' year=2010 director='Christopher Nolan' rating=8.5


### message output alongside parsed Structure

In [45]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(
        json_schema_extra={"description": "The title of the movie"}
    )
    year: int = Field(
        json_schema_extra={"description": "The year the movie was released"}
    )
    director: str = Field(
        json_schema_extra={"description": "The director of the movie"}
    )
    rating: float = Field(
        json_schema_extra={"description": "The movie rating out of 10"}
    )

In [46]:
llm_with_structure=llm.with_structured_output(Movie, include_raw=True)

In [47]:
response = llm_with_structure.invoke("Provie me details about movie iron man 1")

In [48]:
response

{'raw': AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '5354mh2x9', 'function': {'arguments': '{"director":"Jon Favreau","rating":8,"title":"Iron Man","year":2008}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 31, 'prompt_tokens': 282, 'total_tokens': 313, 'completion_time': 0.05465583, 'completion_tokens_details': None, 'prompt_time': 0.048002032, 'prompt_tokens_details': None, 'queue_time': 0.066763248, 'total_time': 0.102657862}, 'model_name': 'llama-3.3-70b-versatile', 'system_fingerprint': 'fp_dae98b5ecb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019d4788-199c-76b2-8efd-f075f25c430f-0', tool_calls=[{'name': 'Movie', 'args': {'director': 'Jon Favreau', 'rating': 8, 'title': 'Iron Man', 'year': 2008}, 'id': '5354mh2x9', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 282, 'output_tokens': 31, 'total_tokens': 31

#### Nested Structure

In [49]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str
    
    
class MovieDetails(BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres:list[str]
    budget:float | None = Field(None, json_schema_extra={"description":"Budget in million USD"})

In [50]:
model_with_structure = llm.with_structured_output(MovieDetails)


In [53]:
response = model_with_structure.invoke("Provide me the details about Dhurandhar: The Revenge")

print(response)

title='Dhurandhar: The Revenge' year=2023 cast=[Actor(name='Rohit Bose Roy', role='Dhurandhar'), Actor(name='Mithila Palkar', role='Shreya')] genres=['Action', 'Thriller'] budget=None
